# Pooled-path test comparison

This notebook does not retrain anything. It copies test metrics that already exist and puts the ones that can be compared in one table.

**Included** (pooled path, original ug/m3, horizons 7 / 14 / 30):

- CNN-LSTM — test `metrics_summary` already stored in `cnn_lstm.ipynb`
- Transformer — test `metrics_summary` already stored in `transformer.ipynb`
- Random Forest — `rf_results_path/random_forest_pooled_path_results.csv`
- XGBoost — `xgb_results_path/xgboost_pooled_path_results.csv`

**Left out:** LSTM / GRU. Direct day-h tree tables. Fit/val scores. The extra 1-day transformer experiment.

Pooled path means every day from t+1 through t+H is scored together, not only day H.


In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

HERE = Path(".")
CNN_NB = HERE / "cnn_lstm.ipynb"
TRANS_NB = HERE / "transformer.ipynb"
RF_CSV = HERE / "rf_results_path" / "random_forest_pooled_path_results.csv"
XGB_CSV = HERE / "xgb_results_path" / "xgboost_pooled_path_results.csv"
RF_PRED = HERE / "rf_results_path"
XGB_PRED = HERE / "xgb_results_path"


## Read CNN-LSTM and transformer test tables from their notebooks

Those notebooks are not executed. The numbers come from the `metrics_summary` output already saved in the ipynb.


In [ ]:
def metrics_summary_from_notebook(nb_path):
    """Pull the stored metrics_summary display (test, 7/14/30)."""
    nb = json.loads(Path(nb_path).read_text())
    plains = []
    for cell in nb["cells"]:
        src = "".join(cell.get("source", []))
        if "metrics_summary" not in src or "metrics_7" not in src:
            continue
        for out in cell.get("outputs", []):
            data = out.get("data") or {}
            plain = data.get("text/plain")
            if not plain:
                continue
            if isinstance(plain, list):
                plain = "".join(plain)
            if "MAE" in plain and "7" in plain and "30" in plain:
                plains.append(plain)
    if not plains:
        raise ValueError(f"No metrics_summary output in {nb_path}")
    text = plains[0]
    rows = []
    for line in text.splitlines():
        parts = line.split()
        if len(parts) == 7 and parts[0] in {"7", "14", "30"}:
            h, mae, rmse, r2, mape, mse, mbe = parts
            rows.append(
                {
                    "Horizon_Days": int(h),
                    "MAE": float(mae),
                    "RMSE": float(rmse),
                    "R2": float(r2),
                    "MAPE": float(mape),
                    "MSE": float(mse),
                    "MBE": float(mbe),
                }
            )
    df = pd.DataFrame(rows)
    if list(df["Horizon_Days"]) != [7, 14, 30]:
        raise ValueError(f"Unexpected horizons in {nb_path}: {df}")
    return df


cnn = metrics_summary_from_notebook(CNN_NB)
cnn.insert(0, "Model", "CNN-LSTM")
cnn["Source"] = "cnn_lstm.ipynb (stored test metrics_summary)"

transformer = metrics_summary_from_notebook(TRANS_NB)
transformer.insert(0, "Model", "Transformer")
transformer["Source"] = "transformer.ipynb (stored test metrics_summary)"

print("CNN-LSTM")
display(cnn)
print("Transformer")
display(transformer)


## Random Forest and XGBoost pooled-path CSVs

These are the stitched 1...H paths, not the day-h tables.


In [ ]:
rf = pd.read_csv(RF_CSV)
xgb = pd.read_csv(XGB_CSV)

tree_cols = [
    "Horizon_Days",
    "Model",
    "MAE",
    "RMSE",
    "R2",
    "MAPE",
    "MSE",
    "MBE",
]
rf_cmp = rf[tree_cols].copy()
rf_cmp["Source"] = str(RF_CSV)
xgb_cmp = xgb[tree_cols].copy()
xgb_cmp["Source"] = str(XGB_CSV)

display(rf_cmp)
display(xgb_cmp)


## Combined comparison table


In [ ]:
comparison_df = pd.concat(
    [cnn, transformer, rf_cmp, xgb_cmp],
    ignore_index=True,
)
comparison_df["Evaluation"] = "pooled_path"
comparison_df["Units"] = "ug/m3 (MAPE in %)"

comparison_df = comparison_df[
    [
        "Model",
        "Horizon_Days",
        "Evaluation",
        "MAE",
        "RMSE",
        "MSE",
        "R2",
        "MAPE",
        "MBE",
        "Source",
        "Units",
    ]
].sort_values(["Horizon_Days", "MAE"]).reset_index(drop=True)

display(comparison_df.round(4))

out_path = HERE / "pooled_path_comparison.csv"
comparison_df.to_csv(out_path, index=False)
print("Saved", out_path.resolve())


## Checks

1. CNN-LSTM / transformer rows match the printed Horizon 7d / 14d / 30d test blocks in those notebooks.
2. RF / XGBoost CSV rows match a fresh pool over `*_predictions_{h}d.csv`.


In [ ]:
def printed_test_metrics(nb_path):
    nb = json.loads(Path(nb_path).read_text())
    text = []
    for cell in nb["cells"]:
        for out in cell.get("outputs", []):
            if out.get("output_type") != "stream":
                continue
            text.append("".join(out.get("text", [])))
    blob = "\n".join(text)
    rows = {}
    for h in (7, 14, 30):
        key = None
        for candidate in (f"Horizon {h}d (test)", f"Horizon {h}d"):
            if candidate in blob:
                key = candidate
                break
        if key is None:
            raise ValueError(f"Missing Horizon {h}d in {nb_path}")
        chunk = blob.split(key, 1)[1]
        got = {}
        for line in chunk.splitlines()[:8]:
            line = line.strip()
            if line.startswith("MAE:"):
                got["MAE"] = float(line.split()[1])
            elif line.startswith("RMSE:"):
                got["RMSE"] = float(line.split()[1])
            elif line.startswith("R2:") or line.startswith("R"):
                got["R2"] = float(line.split()[1])
            elif line.startswith("MAPE:"):
                got["MAPE"] = float(line.split()[1].replace("%", ""))
            elif line.startswith("MSE:"):
                got["MSE"] = float(line.split()[1])
            elif line.startswith("MBE:"):
                got["MBE"] = float(line.split()[1])
        rows[h] = got
    return rows


def path_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float).ravel()
    y_pred = np.asarray(y_pred, dtype=float).ravel()
    mse = mean_squared_error(y_true, y_pred)
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "MSE": mse,
        "RMSE": float(np.sqrt(mse)),
        "R2": float(r2_score(y_true, y_pred)),
        "MAPE": float(
            np.mean(np.abs((y_true - y_pred) / np.maximum(np.abs(y_true), 1e-8))) * 100
        ),
        "MBE": float(np.mean(y_pred - y_true)),
    }


def recompute_pooled(pred_dir, prefix, H):
    dfs = {
        h: pd.read_csv(pred_dir / f"{prefix}_{h}d.csv", parse_dates=["Forecast_Origin"])
        for h in range(1, H + 1)
    }
    common = set.intersection(*[set(dfs[h]["Forecast_Origin"]) for h in range(1, H + 1)])
    y_true, y_pred = [], []
    for origin in sorted(common):
        for h in range(1, H + 1):
            sub = dfs[h].loc[dfs[h]["Forecast_Origin"] == origin]
            y_true.append(sub["Actual_PM25"].iloc[0])
            y_pred.append(sub["Predicted_PM25"].iloc[0])
    return path_metrics(y_true, y_pred)


checks = []

for name, nb_path, subset in [
    ("CNN-LSTM", CNN_NB, cnn),
    ("Transformer", TRANS_NB, transformer),
]:
    printed = printed_test_metrics(nb_path)
    for _, row in subset.iterrows():
        h = int(row["Horizon_Days"])
        for col in ["MAE", "RMSE", "R2", "MSE", "MBE"]:
            ok = abs(row[col] - printed[h][col]) < 0.0015
            checks.append((ok, f"{name} H={h} {col} table={row[col]} print={printed[h][col]}"))
        ok = abs(row["MAPE"] - printed[h]["MAPE"]) < 0.015
        checks.append((ok, f"{name} H={h} MAPE table={row['MAPE']} print={printed[h]['MAPE']}"))

for label, csv_df, pred_dir, prefix in [
    ("Random Forest", rf, RF_PRED, "rf_predictions"),
    ("XGBoost", xgb, XGB_PRED, "xgb_predictions"),
]:
    for H in (7, 14, 30):
        recomputed = recompute_pooled(pred_dir, prefix, H)
        row = csv_df.loc[csv_df["Horizon_Days"] == H].iloc[0]
        for col in ["MAE", "RMSE", "R2", "MAPE", "MSE", "MBE"]:
            ok = abs(float(row[col]) - recomputed[col]) < 1e-6
            checks.append(
                (ok, f"{label} H={H} {col} csv={row[col]} recomputed={recomputed[col]}")
            )

failed = [msg for ok, msg in checks if not ok]
print(f"{len(checks) - len(failed)}/{len(checks)} checks passed")
if failed:
    raise AssertionError("\n".join(failed))
print("All checks passed.")
